<a href="https://colab.research.google.com/github/ProfAI/programmazione-con-python/blob/main/Progetto%20Finale%20-%20Software%20di%20Gestione%20Magazzino/python_progetto_finale.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Vegan store products software

### Libraries import

In [9]:
import json

### Definition of the necessary functions

In [10]:
file_name = "inventory_market.json"


def adding_new_product(inventory, name, quant, sale_price, purch_price):

    '''
    Opens or creates a JSON file containing a list of dictionaries representing an inventory and adds a new product to it.
    Each dictionary in the list represents a product.

    Args:
        inventory (list): list representing the inventory
        name: name of the product to add to the inventory
        quant (int): quantity of the product to add
        sale_price (float): selling price of the product
        purch_price (float): purchase price of the product

    '''

    with open(file_name, 'w+') as json_file:
        inventory.append({"PRODUCT": name,
                        "QUANTITY": quant,
                        "SELLING_PRICE": sale_price,
                        "PURCHASE_PRICE": purch_price})
        json.dump(inventory, json_file, indent=3)



def adding_existing_product(inventory, name, quant):

    '''
    Opens the inventory file and adds the specified quantity to an existing product.

    Args:
        inventory (list): inventory
        name: product name
        quant (int): quantity to add

    '''

    with open(file_name, 'r+') as json_file:
        for el in inventory:

            if el['PRODUCT'] == name:
                el["QUANTITY"] += quant

                json_file.seek(0)
                json_file.truncate()
                json.dump(inventory, json_file, indent=3)



def create_list(inventory):

    ''''
    Create a list of products in stock, removing the purchase price and renaming the selling price as "price".

    Args:
        inventory (list): inventory
    '''

    for el in inventory:
        el['PRICE'] = el.pop('SELLING_PRICE')
        el.pop('PURCHASE_PRICE')



def print_product_list():

    '''
    Print the list of products in stock.

    '''

    with open(file_name) as json_file:
        inv = json.load(json_file)
        collection = inv.copy()
        create_list(collection)
        print("%s\t%s\t%s" % tuple(collection[0].keys()))
        for el in inv:
            print("%s\t\t%i\t\t%s%.2f" % (el['PRODUCT'], el["QUANTITY"], '€',el['PRICE']))



def sale(inventory, sales, name, quant):

    '''
    Removes sold products from the inventory and prints the sale details.

    Args:
        inventory (list): inventory
        sales (list): initially empty list where sales are recorded
        name: name of the product to be sold
        quant: quantity of the product to be sold

    '''

    with open(file_name, 'r+') as json_file:
        for el in inventory:

            if el['PRODUCT'] == name:
                try:
                    assert (quant <= el["QUANTITY"]), 'The indicated quantity is not in stock'
                    el["QUANTITY"] -= quant

                except AssertionError as e:
                    print(e)

                json_file.seek(0)
                json_file.truncate()
                json.dump(inventory, json_file, indent=3)

    sales.append({"PRODUCT": name,
                        "QUANTITY": quant})

    for el in sales:
        for i in inventory:
            if el['PRODUCT'] == i['PRODUCT']:
                el['gross_earnings'] = el["QUANTITY"]* i["SELLING_PRICE"]
                el['net_earnings'] = el["QUANTITY"]* (i["SELLING_PRICE"] - i["PURCHASE_PRICE"])



def help():

    '''
    Print the commands that can be entered into the program.

    '''

    print("The available commands are as follows:\n" \
    "add: add a product to the inventory\n" \
    "list: list the products in the inventory\n" \
    "sale: record a sale\n" \
    "profits: show total profits\n" \
    "help: show available commands\n" \
    "close: exit the program")


def name_exists(list):

    '''
    Allows the user to enter a product name and checks if it exists in a list (e.g., in the warehouse).
    Prompts for the product name until a valid one is entered.

    Args:
        list (list): the list in which to check for the product name.
    '''

    name_exists = False
    while not name_exists:
      try:
        name = input('Name of the product: ')
        assert(name in list), 'Product not in stock'

        name_exists = True
        return name
      except AssertionError as e:
        print(e)



def assert_int():

    '''
    Allows the user to enter a number and verifies that it is an integer. If a float is entered, it converts it to an integer if the decimal part is zero; otherwise, it raises an exception.
    Prompts for a number until an acceptable value is entered.
    '''

    is_float = False
    while not is_float:
        try:
            quant = float(input("Quantity: "))
            assert(quant == int(quant)), "Enter an integer!"
            quant = int(quant)
            is_float = True
            return quant
        except ValueError:
            print(f"Enter a numeric value!")
        except AssertionError as e:
            print(e)



def is_quant_present(inventory, name):

    '''
    Allows the user to enter a product quantity and verifies its availability in the inventory.
    Checks that the entered quantity is an integer and is available in the inventory; otherwise, it raises an exception.
    Prompts the user to enter a quantity until a valid, available amount is provided.

    Args:
        inventory (list): the inventory to check
        name: the name of the product whose quantity is being checked

    '''

    is_quant_present = False
    while not is_quant_present:
      for el in inventory:
        if el['PRODUCT'] == name:
          try:
            quant = assert_int()
            assert(quant <= el["QUANTITY"]), "The indicated quantity is not in stock"

            is_quant_present = True
            return quant
          except AssertionError as e:
             print(e)



def is_float(type):
    '''
    Allows for the input of a number and verifies that it is a float. If an integer is entered, it converts it to a float; if a string is entered, it raises an exception.

    Args:
    type (string): type of number to be entered. Can be 'sale', 'purchase', or 'generic'.

    '''

    is_float = False
    while not is_float:
        try:
            if type == 'generic':
                number = float(input("Quantity: "))
            else:
                number = float(input(f"Price of {type}: "))
            is_float = True
            return number
        except ValueError:
            print('Enter a numeric value!')

### Program code

In [11]:
cmd = None

#list required to record total sales and print them using the "profits" command
total_sales = []


while cmd!="close":

  #Opens the inventory JSON file; creates it if it does not exist.
  try:
      with open(file_name) as json_file:
          inventory = json.load(json_file)
  except FileNotFoundError:
      inventory = []

  #stocks only products with positive quantities
  inventory = [el for el in inventory if el["QUANTITY"] > 0]
  with open(file_name, 'w') as json_file:
    json.dump(inventory, json_file, indent=3)



  cmd = input("Insert a command: ")


  if cmd=="add":
    # adds a product to the inventory; if it is already present, it adds to the quantity

    product_name = input("Name of the product: ")
    product_quant = assert_int()

    products = [el['PRODUCT'] for el in inventory]

    if product_name in products:
        adding_existing_product(inventory,product_name,product_quant)
        print(f"ADDED: {product_quant} x {product_name}")

    elif product_name not in products:

        purchase_price = is_float('purchase')
        sale_price = is_float('sale')

        adding_new_product(inventory, product_name,product_quant,sale_price,purchase_price)
        print(f"ADDED: {product_quant} x {product_name}")



  elif cmd=="list":
    # list all products in the inventory

    try:
      print_product_list()
    except FileNotFoundError:
       print('No products found')
    except IndexError:
       print('No products found')



  elif cmd=="sale":
    # records a sale

    products = [el['PRODUCT'] for el in inventory]

    #empty list where sales are recorded
    sales = []

    product_name = name_exists(products)
    product_quant = is_quant_present(inventory, product_name)

    sale(inventory, sales, product_name, product_quant)
    adding_other = input('Add another product? [yes/no]')


    while adding_other != 'no':

      product_name = name_exists(products)
      product_quant = is_quant_present(inventory, product_name)

      sale(inventory, sales, product_name, product_quant)
      adding_other = input('Add another product? [yes/no]')


    if adding_other == 'no':
      #Print the sale and the total sold.

      print('SALE RECORDED')
      temporary_total = 0
      for el in sales:
        print(f"- {el['QUANTITY']} x {el['PRODUCT']}: €{round(el['gross_earnings'],2)}")

        for i in inventory:
          if el['PRODUCT'] == i['PRODUCT']:
            temporary_total += el["QUANTITY"]*i["SELLING_PRICE"]
      print(f"Total: €{round(temporary_total, 2)}")

    #adds the sales made to the total sales
    total_sales = total_sales + sales


  elif cmd=="profits":
    # shows net and gross profits

    gross_profit = 0
    net_profit = 0

    for el in total_sales:
      gross_profit+= el['gross_earnings']
      net_profit += el['net_earnings']
    print(f"Profit: gross=€{round(gross_profit,2)} net=€{round(net_profit,2)}")



  elif cmd=="help":
    # show the possible commands
    help()


  elif cmd=="close":
    # Say goodbye and stop the program.
    print("Bye bye")

  else:
    # invalid command
    # show help message
    print("invalid command")
    help()

Insert a command: help
The available commands are as follows:
add: add a product to the inventory
list: list the products in the inventory
sale: record a sale
profits: show total profits
help: show available commands
close: exit the program
Insert a command: add
Name of the product: soy milk
Quantity: 20
Price of purchase: 0.80
Price of sale: 1.40
ADDED: 20 x soy milk
Insert a command: add
Name of the product: tofu
Quantity: 10
Price of purchase: 2.20
Price of sale: 4.19
ADDED: 10 x tofu
Insert a command: add
Name of the product: seitan
Quantity: 5
Price of purchase: 3
Price of sale: 5.49
ADDED: 5 x seitan
Insert a command: list
PRODUCT	QUANTITY	PRICE
soy milk		20		€1.40
tofu		10		€4.19
seitan		5		€5.49
Insert a command: sale
Name of the product: soy milk
Quantity: 5
Add another product? [yes/no]yes
Name of the product: tofu
Quantity: 2
Add another product? [yes/no]no
SALE RECORDED
- 5 x soy milk: €7.0
- 2 x tofu: €8.38
Total: €15.38
Insert a command: list
PRODUCT	QUANTITY	PRICE
soy mi